# 8.1 - RAG Fundamentals: Build It From Scratch

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook builds a full Retrieval-Augmented Generation pipeline by hand - chunk, embed, retrieve, augment, generate - in about sixty lines with no framework, then upgrades the two weakest parts (the vector store and the prompt) to their production form. You will see the exact difference between a closed-book LLM that invents a refund policy and an open-book one that answers from your own documents.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the three libraries the notebook uses: the Gemini SDK for embeddings and chat, NumPy for the vector math, and python-dotenv for loading keys. On Colab or a fresh environment, uncomment the line and run it once.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy python-dotenv -q  # noqa


**Explanation**

Environment prep, not logic - one commented pip line you uncomment only when the packages are missing.

**How the code works - step by step**
- **`google-genai`** - the Gemini SDK, used for both `embed_content` (turning text into vectors) and `generate_content` (the LLM answer).
- **`numpy`** - array math for the cosine-similarity retrieval step.
- **`python-dotenv`** - lets you keep the API key in a `.env` file instead of hardcoding it.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

RAG here runs entirely on Gemini, so the pipeline needs one Google API key on the environment before any embed or chat call. This cell reads it from the environment (or a `.env` file) rather than hardcoding it.

> **Needs a Gemini API key** (set GOOGLE_API_KEY from aistudio.google.com)

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A configuration cell that seeds an environment variable, not a model call. `setdefault` only fills GOOGLE_API_KEY if it is not already set, so an existing shell or Colab secret wins over the empty placeholder.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the key name with an empty default; the SDK's `genai.Client()` later reads this variable automatically. Set the real value in your shell or `.env`, never in the code.

**What you'll see:** (no output - environment prep)

## 1 - The failure RAG exists to fix

Before any code, a reconstructed incident: a support bot built on a top-tier LLM was asked about refunds and confidently invented a 30-day no-questions-asked policy that did not exist. This cell is the whole motivation for the lesson in one comment block.

In [ ]:
# Output: the support transcript, reconstructed
#
# User: What is your refund policy?
# Bot:  We offer a 30-day money-back guarantee, no questions asked!
#
# ...except the real policy is 7 days full, 50% up to 30 days, none after.
# The model had never SEEN the company's policy - so it produced a
# confident, fluent, completely INVENTED answer. Support spent the week
# honoring refunds the company never promised.

**Explanation**

A pure-comment cell - no code runs. It reconstructs a support transcript to make the problem concrete: a closed-book LLM answers from training memory, which never contained the company's actual policy, so it produces a fluent, confident, invented answer.

**How the code works - step by step**
- **The `# Output:` block** - shows the bot claiming a 30-day money-back guarantee.
- **The contrast comment** - states the real policy (7 days full, 50 percent to 30 days, none after) and names the root cause: the model had never *seen* the company's data, so it hallucinated.

**In one line:** a model with no access to your documents will invent an answer rather than admit it does not know.

**What you'll see:** No code executes - the cell is a commented transcript showing the invented "30-day money-back guarantee" answer versus the real tiered policy.

## 2 - Chunk documents into retrievable pieces

Nothing can be retrieved until it is cut into pieces. You cannot embed a 50-page PDF as one vector and expect a precise match, so this cell splits text into overlapping word-windows - small enough to be specific, with a shared strip so an idea is never severed at a boundary. This is RAG step 1.

In [ ]:
# RAG STEP 1: CHUNKING
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.split()) >= 20:
            chunks.append({"id": len(chunks), "text": chunk})
    return chunks

doc = "Netsetos is an edtech company in Hyderabad..." * 20
chunks = chunk_text(doc, chunk_size=60, overlap=10)
print(f"Document: {len(doc.split())} words → {len(chunks)} chunks")
print(f"Sweet spot: 200-500 words, 50-word overlap")

# Output:
#   Document: 120 words -> 3 chunks
#   Sweet spot: 200-500 words, 50-word overlap

**Explanation**

A sliding-window splitter over whitespace-tokenized words. The key trick is the stride: each window starts `chunk_size - overlap` words after the previous one, so consecutive chunks share `overlap` words and no sentence is cleanly cut in two.

**How the code works - step by step**
- **`words = text.split()`** - tokenizes on whitespace so windows are measured in words.
- **`range(0, len(words), chunk_size - overlap)`** - the loop stride is the window size minus the overlap, producing the shared boundary strip.
- **`chunk = " ".join(words[i:i+chunk_size])`** - slices out one `chunk_size`-word window.
- **`if len(chunk.split()) >= 20`** - a min-length guard that drops tiny trailing fragments which would only add retrieval noise.
- **the demo call** - builds a repeated document and splits it at `chunk_size=60, overlap=10`.

**In one line:** slide a fixed-width window with overlap, keeping only windows long enough to be useful.

**What you'll see:** Prints the word count mapping to chunk count (`Document: 120 words -> 3 chunks`) and the tuning note `Sweet spot: 200-500 words, 50-word overlap`.

## 3 - The complete RAG pipeline in one class

Here is the whole game: embed every chunk into a vector, keep the vectors in a list, embed the question and match by cosine similarity, paste the top matches into the prompt, and let the model answer from them. Sixty lines, no framework, so nothing is hidden - including how the grounding instruction makes the model refuse to invent.

> **Needs a Gemini API key** (set GOOGLE_API_KEY) - this cell makes real embedding and chat calls.

In [ ]:
# COMPLETE RAG PIPELINE FROM SCRATCH - no frameworks
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()          # reads GOOGLE_API_KEY
EMBED = "gemini-embedding-001"   # MTEB #1; text-embedding-004 is retired
CHAT  = "gemini-2.5-flash"       # gemini-2.0-flash was shut down 2026-06-01

def embed(text: str, task: str) -> np.ndarray:
    # task_type tunes the vector for its job: documents vs queries.
    r = client.models.embed_content(
        model=EMBED, contents=text,
        config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

class SimpleRAG:
    def __init__(self, chunk_size=60, overlap=10, top_k=3):
        assert overlap < chunk_size, "overlap must be smaller than chunk_size"
        self.chunk_size, self.overlap, self.top_k = chunk_size, overlap, top_k
        self.chunks, self.embeddings = [], []

    def add_document(self, text, source="doc"):
        words = text.split()
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk = " ".join(words[i:i + self.chunk_size])
            if len(chunk.split()) < 15:
                continue
            self.chunks.append({"text": chunk, "source": source})
            self.embeddings.append(embed(chunk, "RETRIEVAL_DOCUMENT"))
        print(f"  Indexed: {len(self.chunks)} chunks from '{source}'")

    def retrieve(self, query, k=None):
        k = k or self.top_k
        qe = embed(query, "RETRIEVAL_QUERY")
        # cosine similarity: dot product of unit vectors. Linear scan is fine
        # for a demo; a real corpus uses a vector DB (step 6) for speed.
        scored = [(i, float(qe @ e / (np.linalg.norm(qe) * np.linalg.norm(e))))
                  for i, e in enumerate(self.embeddings)]
        scored.sort(key=lambda x: -x[1])
        return [{"text": self.chunks[i]["text"], "score": s,
                 "source": self.chunks[i]["source"]} for i, s in scored[:k]]

    def query(self, question):
        docs = self.retrieve(question)
        context = "\n\n".join(f"[{d['source']}] {d['text']}" for d in docs)
        prompt = (f"Answer ONLY from this context. If the answer is not in it, say "
                  f"you do not have that information.\n\nContext:\n{context}\n\n"
                  f"Question: {question}\nAnswer:")
        resp = client.models.generate_content(model=CHAT, contents=prompt)
        return {"answer": resp.text.strip(),
                "sources": [{"text": d["text"][:60], "score": f"{d['score']:.3f}"} for d in docs]}

# == BUILD AND TEST ==
rag = SimpleRAG(chunk_size=60, overlap=10, top_k=3)
print("Building RAG...\n")
rag.add_document("Netsetos is an edtech company in Hyderabad offering GenAI, Data "
                 "Science, Cloud courses. Flagship: GenAI Complete Course, 18 "
                 "modules, about 150 hours.", "about.txt")
rag.add_document("Refund policy: Full refund within 7 days. 50 percent from 8-30 "
                 "days. No refunds after 30 days. Processed in 5 business days.", "refund.txt")
rag.add_document("GenAI course: 14999 rupees. Includes lifetime access, Discord, "
                 "weekly live sessions, certificate. EMI via Razorpay.", "pricing.txt")
rag.add_document("Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. "
                 "Python and GCP. 4.8 star rating.", "faq.txt")

print("\nRAG Q&A:\n")
for q in ["What is the refund policy?", "How much does GenAI cost?",
          "Does Netsetos offer blockchain?"]:
    r = rag.query(q)
    print(f"  Q: {q}\n  A: {r['answer'][:120]}\n")

# Output:
#   Indexed: 1 chunks from 'about.txt' ... (4 docs)
#   Q: What is the refund policy?
#   A: Full refund within 7 days, 50 percent from 8-30 days, none after 30 days.
#   Q: Does Netsetos offer blockchain?
#   A: I do not have that information.   <- grounded: it refuses to invent


**Explanation**

The full five-step loop wrapped in one class. `SimpleRAG` indexes documents into parallel `chunks`/`embeddings` lists, retrieves by a linear cosine-similarity scan, and answers with a grounded prompt. Read it as: embed docs once, then for each question embed-match-inject-generate.

**How the code works - step by step**
- **`embed(text, task)`** - one model (`gemini-embedding-001`) embeds both docs and queries, but `task_type` (`RETRIEVAL_DOCUMENT` vs `RETRIEVAL_QUERY`) tunes each vector for its role - a free retrieval-quality lift.
- **`__init__`** - stores the chunking params and two empty parallel lists; asserts overlap is smaller than chunk_size.
- **`add_document`** - slides the same overlapping window as step 2, skips chunks under 15 words, and appends each chunk plus its `RETRIEVAL_DOCUMENT` embedding.
- **`retrieve`** - embeds the query as a `RETRIEVAL_QUERY`, scores every stored vector by cosine similarity (dot product over norms), sorts descending, and returns the top-k.
- **`query`** - joins the retrieved chunks into a context block and wraps them in the grounding prompt ("Answer ONLY from this context... say you do not have that information"), then calls `gemini-2.5-flash`.
- **the build/test block** - indexes four Netsetos docs and asks three questions, including an out-of-scope one.

**In one line:** embed docs -> cosine-match the query -> inject top chunks -> generate a grounded answer.

**What you'll see:** Prints the indexed-chunk count per source, then each question with its answer - the refund question answered from the real tiered policy, and the blockchain question answered "I do not have that information" instead of guessing.

## 4 - RAG with vs without, side by side

One question, two prompts. The only difference is whether the real policy sits in the context. This is the cold-open incident reduced to five lines - the clearest proof that RAG is not a smarter model, just a better-informed one.

> **Needs a Gemini API key** (set GOOGLE_API_KEY) - two live chat calls.

In [ ]:
# RAG vs NO RAG - side by side
from google import genai
client = genai.Client()
MODEL = "gemini-2.5-flash"

q = "What is Netsetos's refund policy?"
context = "Refund: Full within 7 days. 50% from 8-30 days. None after 30 days."

no_rag = client.models.generate_content(model=MODEL, contents=q)
with_rag = client.models.generate_content(
    model=MODEL,
    contents=f"Answer ONLY from context. If missing, say so.\nContext: {context}\nQ: {q}")

print(f"WITHOUT RAG:\n  {no_rag.text.strip()[:150]}\n")
print(f"WITH RAG:\n  {with_rag.text.strip()[:150]}\n")

# Output:
#   WITHOUT RAG:
#     I do not have specific information about Netsetos's refund policy...
#     (or worse: a confident, INVENTED policy)
#   WITH RAG:
#     Full refund within 7 days, 50% from 8-30 days, and no refunds after 30 days.
#   Same model. Same question. The only difference is the retrieved context.


**Explanation**

A controlled A/B test, not new machinery. It fires the identical question at the same model twice - once bare, once with the policy pasted into a grounding prompt - so the difference in answers can only come from the retrieved context.

**How the code works - step by step**
- **`no_rag`** - sends just the question `q`; the model answers from training memory.
- **`with_rag`** - sends the same question wrapped with `context` and an "Answer ONLY from context" instruction.
- **the two prints** - show both answers truncated to 150 chars for direct comparison.

**In one line:** same model, same question, context is the only variable - and it decides correctness.

**What you'll see:** Prints WITHOUT RAG (the model lacks the info or invents a policy) versus WITH RAG (the correct tiered refund answer), underscoring that only the retrieved context changed.

## 5 - Matryoshka embeddings: one vector, a cost/quality dial

The embedding model is the sense of "meaning" your whole system runs on. Matryoshka embeddings are trained so any *prefix* of the vector is still a valid embedding - so one 3072-dim vector doubles as a cheap 256-dim vector: shortlist fast with the small one, rerank precisely with the full one.

> **Needs a Gemini API key** (set GOOGLE_API_KEY) - two embedding calls.

In [ ]:
# Matryoshka: one embedding, a cost/quality dial (google-genai)
from google import genai
from google.genai import types
client = genai.Client()

text = "Netsetos refund policy: full within 7 days."
# Full 3072-dim vector for precise reranking:
full = client.models.embed_content(model="gemini-embedding-001", contents=text).embeddings[0].values
# Truncated 256-dim vector for fast, cheap first-pass search:
small = client.models.embed_content(
    model="gemini-embedding-001", contents=text,
    config=types.EmbedContentConfig(output_dimensionality=256)).embeddings[0].values

print(len(full), len(small))
# Output:
#   3072 256
#   Shortlist with the 256-dim vectors (cheap), then rerank the top hits
#   with the full 3072-dim vectors (precise). Same model, one dial.


**Explanation**

A tiny demonstration of dimension truncation, not a retrieval run. The same text is embedded twice from the same model - once at full width, once capped at 256 dimensions via `output_dimensionality` - to show both are valid embeddings you can mix in a two-stage search.

**How the code works - step by step**
- **`full = ...embed_content(...)`** - the default full 3072-dim vector for precise reranking.
- **`small = ...output_dimensionality=256`** - the same model asked for a truncated 256-dim vector for cheap first-pass search.
- **`print(len(full), len(small))`** - confirms the two lengths.

**In one line:** ask one model for a long vector or a short prefix of it, and use the short one to shortlist, the long one to rerank.

**What you'll see:** Prints `3072 256`, then a comment explaining the two-stage pattern: shortlist with the 256-dim vectors, rerank the top hits with the full 3072-dim vectors.

## 6 - Swap the linear scan for a vector database

The Python-list scan in the pipeline checks every vector for every query - fine for four chunks, hopeless at four million. A vector database is a specialized index that finds the nearest neighbors in milliseconds without comparing against everything. This cell is the retrieve step, productionized.

> **Needs a Gemini API key** (set GOOGLE_API_KEY) - `embed()` from the earlier cell is reused, so run cell 3 first.

In [ ]:
# The vector-DB upgrade: swap the Python-list scan for Chroma
# pip install chromadb
import chromadb
client = chromadb.Client()
col = client.create_collection("netsetos")

# add() stores documents + their embeddings; Chroma builds the index for you.
col.add(ids=["refund", "pricing"],
        documents=["Full refund within 7 days, 50 percent to 30 days.",
                   "GenAI course is 14999 rupees, EMI via Razorpay."],
        embeddings=[embed(d, "RETRIEVAL_DOCUMENT").tolist()
                    for d in ["Full refund...", "GenAI course..."]])

hits = col.query(query_embeddings=[embed("how do I get my money back", "RETRIEVAL_QUERY").tolist()],
                 n_results=1)
print(hits["ids"][0])
# Output:
#   ['refund']
#   Same retrieve step as SimpleRAG, but Chroma's index stays fast at
#   millions of vectors - no linear scan. This is step 2's list, productionized.


**Explanation**

The same embed-and-match step as `SimpleRAG`, but Chroma owns the storage and index. You hand it documents plus their embeddings; it builds the nearest-neighbor index so search stays fast as the corpus grows.

**How the code works - step by step**
- **`chromadb.Client()` + `create_collection`** - spins up an in-memory store and a named collection.
- **`col.add(...)`** - stores two documents with ids and their `RETRIEVAL_DOCUMENT` embeddings (reusing the earlier `embed` helper); Chroma builds the index.
- **`col.query(query_embeddings=...)`** - embeds "how do I get my money back" as a `RETRIEVAL_QUERY` and asks for the single nearest neighbor.
- **`print(hits["ids"][0])`** - shows which document matched.

**In one line:** same cosine retrieval as before, but the index (not a Python loop) does the nearest-neighbor search.

**What you'll see:** Prints `['refund']` - the money-back query matches the refund doc despite sharing no words - with a comment noting this replaces the linear scan and stays fast at millions of vectors.

## 7 - A production RAG prompt: the four layers that stop hallucination

Retrieval found the pages; now the model must answer FROM them, cite them, and handle the awkward cases - conflicting documents and no answer at all. A production RAG prompt is a small contract, not a one-liner. This cell builds that template.

In [ ]:
# A production RAG prompt: the 4 layers that stop hallucination
def build_prompt(question, chunks):
    numbered = "\n".join(f"[{i+1}] ({c['source']}) {c['text']}"
                         for i, c in enumerate(chunks))
    return f"""You are a support assistant. Answer using ONLY the sources below.
Cite the source number [n] for every claim. If two sources disagree, say so.
If the answer is not in the sources, reply exactly: "I do not have that information."

Sources:
{numbered}

Question: {question}
Answer:"""

# Output: (the model now cites and refuses honestly)
#   Q: What is the refund policy?
#   A: Full refund within 7 days; 50 percent from 8-30 days; none after [1].
#   Q: Do you ship internationally?
#   A: I do not have that information.


**Explanation**

A prompt-assembly helper, not a model call. `build_prompt` turns retrieved chunks into a numbered, cited, grounding-instructed prompt string - the four-layer contract (role, grounding rule, numbered sources, explicit no-answer behavior) that a real RAG system sends to the LLM.

**How the code works - step by step**
- **`numbered = "\n".join(...)`** - formats each chunk as `[n] (source) text`, giving the model stable citation handles.
- **the returned f-string** - layers a role ("support assistant"), a grounding rule ("answer using ONLY the sources"), a citation demand ("[n] for every claim"), a conflict rule ("if two sources disagree, say so"), and an exact refusal string for missing answers.

**In one line:** number the sources, then wrap them in role + grounding + citation + refusal rules so the model cites or honestly declines.

**What you'll see:** No live call - `build_prompt` returns a string. The `# Output:` comment illustrates the intended behavior: the refund question answered with a `[1]` citation, and an unsupported shipping question answered "I do not have that information."

You built the whole RAG loop from primitives: chunk_text cuts documents into overlapping windows, SimpleRAG embeds and retrieves by cosine similarity and grounds the answer, and the side-by-side comparison proves the only thing that changed was the retrieved context, not the model. The last cells swapped the two demo shortcuts for their production form - Chroma's index in place of the linear scan, and a four-layer prompt (role, grounding, numbered sources, honest "I do not have that information") in place of the one-line instruction. From here, Lessons 8.2-8.4 reopen document processing, embeddings/vector stores, and the retrieval ladder as full hands-on lessons, and 8.5/8.6 rebuild this same pipeline on LangChain and LlamaIndex.